# Guide d'utilisation du plugin de backtest factoriel

## 1. Charger le plugin

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\dev\factor_backtest")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import BacktestEngine
import func
import factor_config

importlib.reload(BacktestEngine)
importlib.reload(factor_config)
importlib.reload(func)

from func import (
    RECOMMENDED_PERIOD_BREAKPOINTS,
    calculate_benchmark_performance,
    export_backtest_results,
    load_backtest_data,
    plot_variable_missingness,
    plot_performance_comparison,
    prepare_performance_comparisons_by_period,
    test_composite_signals,
    test_incremental_signals,
    test_unitary_signals,
)
from factor_config import (
    FACTOR_FAMILIES,
    RAW_VARIABLES as CATALOG_RAW_VARIABLES,
    factor_columns,
    make_signal_dimensions,
    signal_options,
)

print("Plugin chargé")

## 2. Choisir les familles et préparer la configuration

In [ ]:
DATA_DIR = PLUGIN_DIR / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
BENCHMARK = "STOXX EUROPE 600"
START_DATE = "2007-12-01"
N_JOBS = 12
PERIOD_BREAKPOINTS = list(RECOMMENDED_PERIOD_BREAKPOINTS)
EXPORT_ROOT = PLUGIN_DIR / "exports"
EXPORT_NAME = "factor_research_run"
EXPORT_DIR = EXPORT_ROOT / EXPORT_NAME
EXPORT_HTML = False
EXPORT_PNG = False

SELECTED_FAMILIES = ("growth",)
FAMILY_VARIABLES = factor_columns(*SELECTED_FAMILIES)
RAW_VARIABLES = list(FAMILY_VARIABLES)
EXTRA_COMPOSITE_VARIABLES = ["FCF Conversion", "Net Debt to Ebit"]
LOAD_VARIABLES = list(dict.fromkeys([*RAW_VARIABLES, *EXTRA_COMPOSITE_VARIABLES]))
list_noire_path = None

In [ ]:
# Pour tout tester : RAW_VARIABLES = list(CATALOG_RAW_VARIABLES)
family_sizes = {name: len(columns) for name, columns in FACTOR_FAMILIES.items()}
display(pd.Series(family_sizes, name="Nombre de variables"))
display(pd.Series(RAW_VARIABLES, name="Variables sélectionnées"))

## 3. Charger les données et calculer une fois le benchmark

In [ ]:
screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=LOAD_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

print(f"screen : {screen.shape}, returns : {returns.shape}")

### Filtrer les variables selon les données manquantes

In [ ]:
MISSINGNESS_THRESHOLD = 30  # 30 % de données manquantes au maximum.

missingness_result = plot_variable_missingness(
    screen=screen,
    families=SELECTED_FAMILIES,
    threshold=MISSINGNESS_THRESHOLD,
    bench=BENCHMARK,
    show_plot=True,
)
RAW_VARIABLES = missingness_result["selected_variables"]

display(missingness_result["summary"])
print(f"Variables retenues pour le backtest : {len(RAW_VARIABLES)}")
print(f"Variables exclues : {missingness_result['excluded_variables']}")
if not RAW_VARIABLES:
    raise ValueError("Aucune variable ne respecte le seuil de données manquantes.")

### Paramètres communs à tous les tests

In [ ]:
BENCH_PERF = calculate_benchmark_performance(
    screen=screen, returns=returns, bench=BENCHMARK, start_date=START_DATE
)

MONTHLY_BASE_CACHE = {}

RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": 0.13,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": MONTHLY_BASE_CACHE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": True,
    "build_figure": True,
}

In [ ]:
display(BENCH_PERF.rename("Benchmark").to_frame().tail())
display(pd.Series({key: value for key, value in RUN_OPTIONS.items() if key != "bench_perf"}))

## 4. Tester les signaux unitaires

In [ ]:
UNITARY_DIMENSIONS = make_signal_dimensions(periods=(1, 3, 6, 12))
# Chaque variable de growth est déjà incluse dans RAW_VARIABLES.
# UNITARY_DIMENSIONS = make_signal_dimensions(periods=(1, 3, 6, 12))

unitary_batch = test_unitary_signals(
    screen=screen,
    returns=returns,
    signal_config=RAW_VARIABLES,
    list_noire_path=list_noire_path,
    dimensions=UNITARY_DIMENSIONS,
    **RUN_OPTIONS,
)
screen = unitary_batch["screen"]
unitary_export = export_backtest_results(
    results={"unitary": unitary_batch},
    output_dir=EXPORT_ROOT,
    export_name=EXPORT_NAME,
    export_html=EXPORT_HTML,
    export_png=EXPORT_PNG,
)

### Colonnes dérivées

In [ ]:
derived_columns = [
    column for column in screen.columns
    if ("__over__" in column or "__pct_" in column
        or "__diff_" in column or "__rank_diff_" in column)
]
derived_columns

## 5. Effectuer une analyse incrémentale

In [ ]:
BASELINE_CONFIG = {
    "Revenue 5Y CAGR": signal_options(level=1.0, pct_1=0.5),
}
CANDIDATE_CONFIG = {
    "FCF Conversion": signal_options(level=1.0, diff_3=0.5),
    "Net Debt to Ebit": signal_options(
        higher_is_better=False, level=1.0, rank_diff_6=0.5
    ),
}

incremental_batch = test_incremental_signals(
    screen=screen,
    returns=returns,
    baseline_config=BASELINE_CONFIG,
    candidate_config=CANDIDATE_CONFIG,
    list_noire_path=list_noire_path,
    **RUN_OPTIONS,
)
screen = incremental_batch["screen"]
incremental_export = export_backtest_results(
    results={"incremental": incremental_batch},
    output_dir=EXPORT_ROOT,
    export_name=EXPORT_NAME,
    export_html=EXPORT_HTML,
    export_png=EXPORT_PNG,
)

## 6. Construire et backtester plusieurs scores composites

In [ ]:
COMPOSITE_CONFIGS = {
    "Croissance mixte": {
        "Revenue 5Y CAGR": signal_options(level=1.0, pct_3=0.5),
        "FCF Conversion": signal_options(level=1.0, diff_6=0.5),
        "Net Debt to Ebit": signal_options(
            higher_is_better=False, level=1.0, rank_diff_12=0.5
        ),
    },
    "Croissance variations": {
        "Revenue 5Y CAGR": signal_options(pct_12=1.0),
        "FCF Conversion": signal_options(diff_3=1.0),
        "Net Debt to Ebit": signal_options(
            higher_is_better=False, level=0.5, rank_diff_6=0.5
        ),
    },
}

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=list_noire_path,
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
composite_export = export_backtest_results(
    results={"composite": composite_batch},
    output_dir=EXPORT_ROOT,
    export_name=EXPORT_NAME,
    export_html=EXPORT_HTML,
    export_png=EXPORT_PNG,
)

## 7. Reconstruire et combiner les performances

In [ ]:
backtest_metrics = pd.read_csv(EXPORT_DIR / "backtest_metrics.csv")
display(backtest_metrics)
display(backtest_metrics[["period_label", "test_path", "robust_score"]])
display(backtest_metrics[["period_label", "test_path", "robust_score"]])
display(backtest_metrics[["period_label", "test_path", "robust_score"]])

# Pour une lecture rapide des recettes composites :
display(backtest_metrics.loc[
    backtest_metrics["scope"].eq("total")
    & backtest_metrics["test_type"].eq("composite"),
    ["test_name", "composition_recipe"],
])

In [ ]:
COMPARISON_MAX_TESTS = 8  # Augmentez cette valeur seulement si chaque figure reste lisible.

comparisons_by_period = prepare_performance_comparisons_by_period(
    export_dir=EXPORT_DIR,
    max_tests=COMPARISON_MAX_TESTS,
    period_breakpoints=PERIOD_BREAKPOINTS,
)

comparison_figures = {}
for period_id, comparison in comparisons_by_period.items():
    comparison_figure = plot_performance_comparison(
        performance=comparison["performance"],
        ratios=comparison["ratios"],
        benchmark_column="Benchmark",
        title="Comparaison des performances",
        save_path=None,
        show_plot=True,
        rebase=True,
        show_worst_performance=False,
        period_definitions=comparison["period_definitions"],
        default_period_id=period_id,
    )
    comparison_figures[period_id] = comparison_figure

prompt_metrics = pd.read_csv(EXPORT_DIR / "backtest_metrics.csv")
print("\n" + "=" * 100)
print(f"EXPORT_DIR : {EXPORT_DIR}")
print("Données complètes des comparaisons, prêtes à copier dans un prompt")
print("=" * 100)

for period_id, comparison in comparisons_by_period.items():
    period = comparison["period"]
    selected_top = [
        (label, test_path)
        for label, (test_path, portfolio)
        in comparison["performance_selection"].items()
        if portfolio == "Top"
    ]
    print("\n" + "#" * 100)
    print(f"PERIOD_ID : {period_id}")
    print(f"Période : {period['label']}")
    print(f"Début réel : {period.get('start')}")
    print(f"Fin réelle : {period.get('end')}")
    if not selected_top:
        print("Aucune performance Top sélectionnée.")
        continue

    selected_labels = [label for label, _ in selected_top]
    selected_paths = [test_path for _, test_path in selected_top]
    period_metrics = prompt_metrics.loc[
        prompt_metrics["period_id"].astype(str).eq(str(period_id))
        & prompt_metrics["test_path"].isin(selected_paths)
    ].copy()
    if not period_metrics.empty:
        period_metrics["_selection_order"] = period_metrics["test_path"].map(
            {test_path: index for index, test_path in enumerate(selected_paths)}
        )
        period_metrics = period_metrics.sort_values("_selection_order").drop(
            columns="_selection_order",
        )
    print("\nFacteurs Top sélectionnés (ordre décroissant du Robust Score) :")
    print(pd.DataFrame({
        "label": selected_labels,
        "test_path": selected_paths,
    }).to_csv(index=False))
    print("Metrics complètes des facteurs sélectionnés (CSV) :")
    print(period_metrics.to_csv(index=False))

    performance = comparison["performance"].copy()
    performance.index = pd.to_datetime(performance.index, errors="coerce")
    performance = performance.loc[performance.index.notna()]
    performance_mask = pd.Series(True, index=performance.index)
    if period.get("start") is not None and pd.notna(period.get("start")):
        performance_mask &= performance.index >= pd.to_datetime(period["start"])
    if period.get("end") is not None and pd.notna(period.get("end")):
        performance_mask &= performance.index <= pd.to_datetime(period["end"])
    performance_columns = [
        label for label in selected_labels if label in performance.columns
    ]
    if "Benchmark" in performance.columns:
        performance_columns.append("Benchmark")
    period_performance = performance.loc[performance_mask, performance_columns]
    period_performance = func._rebase_frame(period_performance, base_value=100.0)
    print("Performances cumulées rebasées à 100, comme dans le graphique (CSV) :")
    print(period_performance.to_csv(index=True, date_format="%Y-%m-%d"))

    ratios = comparison["ratios"].copy()
    ratios.index = pd.to_datetime(ratios.index, errors="coerce")
    ratios = ratios.loc[ratios.index.notna()]
    ratios_mask = pd.Series(True, index=ratios.index)
    if period.get("start") is not None and pd.notna(period.get("start")):
        ratios_mask &= ratios.index >= pd.to_datetime(period["start"])
    if period.get("end") is not None and pd.notna(period.get("end")):
        ratios_mask &= ratios.index <= pd.to_datetime(period["end"])
    ratio_columns = [
        column for column in ratios.columns
        if any(str(column).startswith(f"{label} / ") for label in selected_labels)
    ]
    period_ratios = ratios.loc[ratios_mask, ratio_columns]
    period_ratios = func._rebase_frame(period_ratios, base_value=1.0)
    print("Ratios rebasés à 1, comme dans le graphique (CSV) :")
    print(period_ratios.to_csv(index=True, date_format="%Y-%m-%d"))